# Attack2Defense: all experiments on a Colab GPU

This notebook trains and evaluates every model used in the report.

1. **Runtime -> Change runtime type -> T4 GPU** (any GPU works).
2. Run the cells from top to bottom. Total time on a T4 is roughly 2-3 hours.
3. The last cell downloads `results.zip`. Put it in the project folder and run `python scripts/build_report.py --results results.zip`.

Results are written to Google Drive as they are produced. If Colab disconnects, run the notebook again: finished runs are skipped and an interrupted run resumes from its last epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Project code and result storage

Upload `attack2defense_project.zip` when asked. Outputs go to `MyDrive/Attack2Defense_results`, so they survive a disconnect.

In [ ]:
import os
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")

PROJECT = Path("/content/Attack2Defense")
RESULTS = Path("/content/drive/MyDrive/Attack2Defense_results")
RESULTS.mkdir(parents=True, exist_ok=True)

if not (PROJECT / "attack2defense").exists():
    uploaded = files.upload()
    archive = next(name for name in uploaded if name.endswith(".zip"))
    !unzip -q -o "{archive}" -d "{PROJECT}"

os.chdir(PROJECT)
if not Path("outputs").exists():
    os.symlink(RESULTS, "outputs")
print("Project:", Path.cwd())
print("Outputs:", Path("outputs").resolve())

In [ ]:
!pip -q install -r requirements.txt pytest torchattacks
!python -m pytest tests -q --ignore=tests/api

## 2. Settings

Every model is a CIFAR ResNet-18 trained on all 10 classes with the same recipe: SGD, learning rate 0.1, weight decay 5e-4, a OneCycle schedule, and batch 128. Only the training inputs differ.

| Run | Training inputs |
| --- | --- |
| `resnet18_clean` | Ordinary images (no defense) |
| `resnet18_fgsm` | One-step FGSM with random start, epsilon 2/255 (fast adversarial training) |
| `resnet18_pgd` | 5-step PGD, epsilon 2/255 (PGD adversarial training) |

In [ ]:
EPOCHS = 30
PGD_EPOCHS = 30
EPSILON = "2/255"
DATA_DIR = "/content/data"

COMMON = (
    f"--model resnet18 --optimizer sgd --lr 0.1 --weight-decay 5e-4 --scheduler onecycle "
    f"--batch-size 128 --epsilon {EPSILON} --num-workers 2 --data-dir {DATA_DIR} --quiet"
)


def train(name: str, mode: str, epochs: int, extra: str = "") -> None:
    """Train one run, skipping it if finished and resuming it if interrupted."""

    out = Path("outputs") / name
    if (out / "test_eval" / "summary.csv").exists():
        print(f"{name}: already finished, skipping.")
        return
    resume = f"--resume {out / 'last.pt'}" if (out / "last.pt").exists() else ""
    command = (
        f"python -m attack2defense train {COMMON} --mode {mode} --epochs {epochs} "
        f"--output-dir {out} {resume} {extra}"
    )
    print(command)
    get_ipython().system(command)


def evaluate(name: str) -> None:
    """Full test set at 2/255, plus the 2/255, 4/255, 8/255 sweep."""

    out = Path("outputs") / name
    if (out / "test_eval" / "sweep.csv").exists():
        print(f"{name}: sweep already done, skipping.")
        return
    command = (
        f"python -m attack2defense evaluate --checkpoint {out / 'best.pt'} "
        f"--output-dir {out / 'test_eval'} --epsilon {EPSILON} --pgd-steps 10 "
        f"--sweep --sweep-per-class 1000 --batch-size 256 --num-workers 2 "
        f"--data-dir {DATA_DIR} --quiet"
    )
    print(command)
    get_ipython().system(command)

## 3. Training

Each run logs one line per epoch and then evaluates its best checkpoint on the 10,000-image test set.

In [ ]:
train("resnet18_clean", "clean", EPOCHS)

In [ ]:
train("resnet18_fgsm", "fgsm", EPOCHS)

In [ ]:
train("resnet18_pgd", "pgd", PGD_EPOCHS, extra="--pgd-steps 5")

## 4. Attack-strength sweep

Every model is attacked at 2/255, 4/255, and 8/255 on the full test set.

In [ ]:
for run in ("resnet18_clean", "resnet18_fgsm", "resnet18_pgd"):
    evaluate(run)

## 5. Side-by-side comparison and last week's zero-shot benchmark

In [ ]:
!python -m attack2defense compare \
    outputs/resnet18_clean/test_eval/summary.csv \
    outputs/resnet18_fgsm/test_eval/summary.csv \
    outputs/resnet18_pgd/test_eval/summary.csv \
    --output outputs/comparison.png

In [ ]:
if not Path("outputs/zeroshot/zeroshot_summary.csv").exists():
    !python -m attack2defense benchmark --num-images 500 --batch-size 32 --data-dir {DATA_DIR} --quiet

## 6. Extra experiments (certified defense, Imagenette, AutoAttack/CW)

These are the Should Have / stretch-goal runs. Each step skips work that already exists on Drive.

- **Gaussian ResNet-18** is trained with `N(0, 0.25)` noise, then certified (Cohen et al.). The certified curve is *not* empirical FGSM/PGD accuracy.
- **Imagenette** trains a clean baseline and a PGD-defended ResNet-18 at 128px, then runs the same epsilon sweep.
- **CW and AutoAttack** evaluate the existing CIFAR-10 checkpoints on 1,000 test images.

In [ ]:
train("resnet18_gaussian", "gaussian", EPOCHS, extra="--sigma 0.25")

if not Path("outputs/resnet18_gaussian/certify/certify.csv").exists() and Path("outputs/resnet18_gaussian/best.pt").exists():
    !python -m attack2defense certify --checkpoint outputs/resnet18_gaussian/best.pt \
        --output-dir outputs/resnet18_gaussian/certify --sigma 0.25 --n0 100 --n 1000 \
        --max-per-class 50 --batch-size 256 --data-dir {DATA_DIR} --quiet

train("imagenette_resnet18_clean", "clean", EPOCHS, extra="--dataset imagenette")
train("imagenette_resnet18_pgd", "pgd", PGD_EPOCHS, extra="--dataset imagenette --pgd-steps 5")
for run in ("imagenette_resnet18_clean", "imagenette_resnet18_pgd"):
    evaluate(run)

for run in ("resnet18_clean", "resnet18_fgsm", "resnet18_pgd"):
    extra_dir = Path("outputs") / run / "extra_attacks"
    if extra_dir.joinpath("extra_attacks.csv").exists():
        print(f"{run}: extra attacks already done, skipping.")
        continue
    ckpt = Path("outputs") / run / "best.pt"
    if not ckpt.exists():
        print(f"{run}: no checkpoint, skipping extra attacks.")
        continue
    !python -m attack2defense benchmark-attacks --checkpoint {ckpt} \
        --output-dir {extra_dir} --attacks cw autoattack --max-images 1000 \
        --batch-size 32 --data-dir {DATA_DIR} --quiet

## 7. Quick look and download

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("outputs/comparison.csv"))
display(Image("outputs/comparison.png"))

In [ ]:
# Checkpoints are left out to keep the download small; they stay on Drive.
!cd outputs && zip -q -r /content/results.zip . -x "*.pt"
files.download("/content/results.zip")